# Predicting the video

In [2]:
import numpy as np
import cv2
import os
from keras.models import load_model
from collections import deque

In [4]:
import numpy as np
import cv2
import os
from keras.models import load_model
from collections import deque

def print_results(video, limit=None):
    if not os.path.exists('output'):
        os.mkdir('output')

    print("Loading model ...")
    model = load_model('Modelnew.h5')
    Q = deque(maxlen=128)
    vs = cv2.VideoCapture(video)
    
    if not vs.isOpened():
        print("Error: Could not open video.")
        return

    writer = None
    (W, H) = (None, None)

    while True:
        (grabbed, frame) = vs.read()
        if not grabbed:
            break

        if W is None or H is None:
            (H, W) = frame.shape[:2]

        output = frame.copy()
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (128, 128)).astype("float32")
        frame = frame.reshape(128, 128, 3) / 255

        preds = model.predict(np.expand_dims(frame, axis=0))[0]
        Q.append(preds)

        results = np.array(Q).mean(axis=0)
        i = (preds > 0.50)[0]
        label = i

        text_color = (0, 255, 0)  # Default: green
        if label:
            text_color = (0, 0, 255)  # Red

        text = f"Violence: {label}"
        FONT = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(output, text, (35, 50), FONT, 1.25, text_color, 3)

        if writer is None:
            fourcc = cv2.VideoWriter_fourcc(*"MJPG")
            writer = cv2.VideoWriter("output/v_output.avi", fourcc, 30, (W, H), True)

        writer.write(output)
        cv2.imshow("Output", output)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break

    print("[INFO] cleaning up...")
    if writer is not None:
        writer.release()
    vs.release()
    cv2.destroyAllWindows()


In [1]:
V_path = "V_1.mp4"
NV_path = "nonv.mp4"

In [8]:
print_results(V_path)

Loading model ...
1/1 [==============================] - 0s 33ms/step
[INFO] cleaning up...
